<a href="https://colab.research.google.com/github/Don4141/ProductPriceAI/blob/main/notebooks/Modal_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook implements a Retrieval-Augmented Generation (RAG) pipeline for product price prediction using a training dataset of 800,000 Amazon products. Product summaries are transformed into 384-dimensional semantic embeddings using a Sentence Transformer and stored in ChromaDB alongside their categories and prices. For an unseen product, the pipeline retrieves five semantically similar products and provides their descriptions and prices as contextual information to GPT-5.1 to generate a price estimate. The RAG-based pricing model is subsequently evaluated using held-out test data.

#### Deploy multiple models to production including proprietary LLMs on Modal, RAG workflow with Frontier models, and Neural Network Models.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
os.chdir("/content")
#Clone repo after a new Colab runtime as the previous will be erased
!git clone https://github.com/Don4141/ProductPriceAI.git

#Set repository as working directory
os.chdir("/content/ProductPriceAI")

#Verify
print("Repo exist", os.path.exists("/content/ProductPriceAI"))
print("Current working directory", os.getcwd())

Cloning into 'ProductPriceAI'...
remote: Enumerating objects: 306, done.
remote: Counting objects: 100% (306/306), done.
remote: Compressing objects: 100% (253/253), done.
remote: Total 306 (delta 156), reused 144 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (306/306), 17.99 MiB | 22.80 MiB/s, done.
Resolving deltas: 100% (156/156), done.
Repo exist True
Current working directory /content/ProductPriceAI


In [3]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1 transformers==4.57.6

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
!pip uninstall -y gradio gradio-client diffusers

Found existing installation: gradio 6.31.0
Uninstalling gradio-6.31.0:
  Successfully uninstalled gradio-6.31.0
Found existing installation: gradio_client 2.7.2
Uninstalling gradio_client-2.7.2:
  Successfully uninstalled gradio_client-2.7.2
Found existing installation: diffusers 0.42.1
Uninstalling diffusers-0.42.1:
  Successfully uninstalled diffusers-0.42.1


In [13]:
#Install the list of required Python packages
%pip install -q -r requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [14]:
#Manage imports
import re
import math
import numpy as np
import torch
import wandb
import transformers
import trl
import peft
import bitsandbytes
import huggingface_hub
from tqdm import tqdm
import locale
import logging
from pricer.items import Item
from pricer.evaluator import evaluate, Tester
from tqdm.notebook import tqdm
from IPython.display import display, Image
from pathlib import Path
from transformers import AutoTokenizer
from google.colab import userdata
from huggingface_hub import login
import transformers
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
import matplotlib.pyplot as plt
from agents.preprocessor import Preprocessor
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from agents.items import Item

print("All LLM training libraries imported successfully!")

All LLM training libraries imported successfully!


In [9]:
#Load API credentials
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY", "GITHUB_TOKEN",
            "HF_TOKEN", "MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"]:
  os.environ[key] = userdata.get(key)

In [10]:
#Verify if API keys are loaded
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY", "GITHUB_TOKEN",
            "HF_TOKEN", "MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"]:
    print(f"{key}: {'loaded' if os.getenv(key) else 'Missing'}")

OPENAI_API_KEY: loaded
ANTHROPIC_API_KEY: loaded
GROQ_API_KEY: loaded
GITHUB_TOKEN: loaded
HF_TOKEN: loaded
MODAL_TOKEN_ID: loaded
MODAL_TOKEN_SECRET: loaded


In [11]:
#Verify authentication with Modal
!modal secret list

                                    Secrets                                     
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Name               ┃ Created at           ┃ Created by ┃ Last used at        ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ huggingface-secret │ 2026-11-19 10:39 UTC │ don4141    │ 2026-11-24 05:19    │
│                    │                      │            │ UTC                 │
└────────────────────┴──────────────────────┴────────────┴─────────────────────┘


In [13]:
#Initialize the preprocessor and transform a sample product description into a concise,
#structured summary using `groq/openai/gpt-oss-20b` as the default LLM.

preprocessor = Preprocessor()
text = preprocessor.preprocess("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")
print(text)

Title: Quadcast HyperX Condenser Microphone  
Category: Audio Equipment  
Brand: HyperX  
Description: A USB-C condenser mic delivering crystal‑clear sound for streaming, podcasting, and gaming.  
Details: Features dual‑polar pattern switching, built‑in headphone monitoring, and on‑board mute control.


In [89]:
#Deploy model
!modal deploy /content/ProductPriceAI/other/pricer_service2.py

⠦ Creating objects...
⠏ Creating objects...
└── ⠸ Creating mount /content/ProductPriceAI/other/pricer_service2.py: Uploaded 
⠹ Creating objects...
└── ⠦ Creating mount /content/ProductPriceAI/other/pricer_service2.py: Uploaded 
⠴ Creating objects...
└── ⠏ Creating mount /content/ProductPriceAI/other/pricer_service2.py: 
⠇ Creating objects...
├── ✓ Uploaded /content/ProductPriceAI/other/pricer_service2.py
⠙ Creating objects...
├── ✓ Uploaded /content/ProductPriceAI/other/pricer_service2.py
⠼ Creating objects...
├── ✓ Uploaded /content/ProductPriceAI/other/pricer_service2.py
└── ✓ Created Function Pricer.*.
✓ Created objects.
├── ✓ Uploaded /content/ProductPriceAI/other/pricer_service2.py
└── ✓ Created Function Pricer.*.
✓ App deployed in 3.037s! 🎉

View Deployment: ]8;id=426966;https://modal.com/apps/don4141/main/deployed/pricer-service\https://modal.com/apps/don4141/main/deployed/pricer-service]8;;\


In [15]:
from agents.specialist_agent import SpecialistAgent

In [16]:
agent = SpecialistAgent()

In [17]:
agent.price("iPhone 13 Pro Max")

655.0

##### Load the preprocessed products dataset from Hugging Face. This has the 819,998 preprocessed product descriptions.

In [15]:
#Load the preprocessed product dataset from Hugging Face
username = "sadjei65320"
dataset = f"{username}/items_full"

#Retrieve the training, validation, and test datasets
train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

README.md:   0%|          | 0.00/744 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/237M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/3.00M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/3.00M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/799998 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loaded 799,998 training items, 10,000 validation items, 10,000 test items


Initialize a persistent ChromaDB vector database and load a pretrained Sentence Transformer model to generate semantic embeddings from product descriptions.

In [16]:
#Define the path for the persistent ChromaDB vector store
DB = "products_vectorstore"

#Initialize the persistent ChromaDB client
client = chromadb.PersistentClient(path=DB)

#Load pretrained sentence embedding model
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Desmonstrate that the text embedding generation works. Convert a sample text into a 384-dimensional numerical embedding and inspect its shape and values.

In [17]:
#Generate a semantic embedding from a sample text. Pass in a list of texts, get back a numpy array of vectors
vector = encoder.encode(["A proficient AI engineer who has almost reached the finale of AI Engineering Core Track!"])[0]
print(vector.shape)
vector

(384,)


array([-5.68234064e-02, -6.70465380e-02,  4.41129729e-02,  5.98604744e-03,
       -2.28948798e-02, -2.95300651e-02,  5.56369089e-02,  3.42665501e-02,
       -1.08529374e-01, -3.81690674e-02, -7.43872449e-02, -1.03664309e-01,
        1.69147570e-02,  1.33930123e-03, -6.86191395e-02,  8.99353474e-02,
       -1.45186167e-02, -2.43885275e-02,  4.21856157e-03, -9.62992907e-02,
       -2.51799505e-02,  4.60676029e-02,  4.95349104e-03, -3.88679989e-02,
        1.07718504e-03,  6.82337284e-02, -1.13860043e-02, -5.83416149e-02,
       -1.03801368e-02, -1.74953155e-02, -1.86478384e-02,  4.07059258e-03,
        1.59438215e-02,  6.49722740e-02,  3.71175930e-02,  2.78224945e-02,
       -4.41944972e-02, -2.34372616e-02,  9.71035659e-02, -5.06139286e-02,
       -1.93864945e-02, -3.83471586e-02,  4.76067066e-02, -3.36107202e-02,
        5.08286953e-02,  3.57934758e-02,  2.91817891e-03, -1.06529161e-01,
        4.07211669e-02, -5.85440779e-04, -1.05607457e-01, -1.03584342e-01,
        3.71123515e-02, -

Generate semantic embeddings for all 799,998 training product summaries in batches of 1,000 and store them in ChromaDB alongside their descriptions, categories, prices, and unique identifiers. Subset and visualize few of the product embeddings.

In [ ]:
#Generate and store product embeddings
collection_name = "products"
collection = client.create_collection(collection_name)
for i in tqdm(range(0, len(train), 1000)):
    documents = [item.summary for item in train[i: i+1000]]
    #Convert product summaries into semantic embeddings
    vectors = encoder.encode(documents).astype(float).tolist()
    #Extract product categories and prices as metadata
    metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
    ids = [f"doc_{j}" for j in range(i, i+1000)]
    ids = ids[:len(documents)]
    #Store embeddings, summaries, and metadata in ChromaDB
    collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)
collection = client.get_or_create_collection(collection_name)

In [ ]:
#Subset and visualize vectors/embeddings of 10,000 products from the collections stored in chroma DB
MAXIMUM_DATAPOINTS = 10_000

In [ ]:
#Define product categories and assign colors to facilitate visual differentiation
CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [ ]:
#Retrieve product embeddings, summaries, and metadata
result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAXIMUM_DATAPOINTS)
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [metadata['category'] for metadata in result['metadatas']]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

In [ ]:
#Apply t-distributed Stochastic Neighbor Embedding (t-SNE) to project the 384-dimensional product embeddings into two dimensions for visualization
tsne = TSNE(n_components=2, random_state=42)
#Reduce the dimensionality of product embeddings from 384 to 2
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
#Generate 2D scatter plot of the reduced product embeddings, color-coded by category, to explore semantic clustering and relationships among products.
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vectorstore Visualization',
    xaxis_title="t-SNE Dimension 1",
    yaxis_title="t-SNE Dimension 2",
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
#Try with three-dimensional dimensionality reduction
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
#Define a reusable function that takes an product discription and transform it
#into a semantic embedding using the pretrained Sentence Transformer
def vector(item):
    return encoder.encode(item.summary)

In [ ]:
#Define another reusable function that when given a product discription, transforms it into a semantic
# embedding using the pretrained Sentence Transformer, then queries the ChromaDB using the product embedding
#and retrieves five nearest (most semantically similar) product summaries and their associated prices
def find_similars(item):
    vec = vector(item)
    #Search the vector database for the five nearest products
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)

    #Extract and retrieve product summaries and prices
    documents = results['documents'][0][:]
    prices = [m['price'] for m in results['metadatas'][0][:]]
    return documents, prices

In [ ]:
#Test the semantic similarity retrieval function above by applying it to the first item in the test set.
find_similars(test[0])

Retrieve the five most semantically similar training products and their associated prices for the sample product.

---

Construct retrieval-augmented context. Format the retrieved product summaries and their known prices into contextual information that can guide the LLM in estimating the price of an unseen product.

In [ ]:
# We need to give some context to GPT-5.1 by selecting 5 products with similar descriptions
#Create a function such that when given a context which is the description of a product,
#it will search through the vectorstore created and return related products and their associated prices.

# Construct contextual information from retrieved products and prices
def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    # Append each retrieved product summary and its known price
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

Inspect the retrieved context: Retrieve similar products for a sample test item and inspect the formatted contextual information before passing it to the LLM.

In [ ]:
# Retrieve similar products and their associated prices
documents, prices = find_similars(test[0])
# Generate and display the retrieval-augmented context
print(make_context(documents, prices))

In [ ]:
#Turn above function into an open AI style messages
def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [ ]:
documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices)[0]['content'])

In [ ]:
# The function for gpt-5.1
#Create a function that calls GPT5.1 in RAG mode. This function will take the description of an item, find related items from the RAG created
#The call GPT 5.1 and pass in the item description and the retrieved related items as extra context
def gpt_5__1_rag(item):
    documents, prices = find_similars(item)
    response = completion(model="gpt-5.1", messages=messages_for(item, documents, prices), reasoning_effort="none", seed=42)
    return response.choices[0].message.content

In [ ]:
# How much does our favorite distortion pedal cost?
test[0].price

In [ ]:
#Thi run will first look up in the Chroma database, find similar items and then make a call to GPT 5.1 in the cloud to come up with an estimate of the product
gpt_5__1_rag(test[0])

In [ ]:
#Apply that to the entire test set
evaluate(gpt_5__1_rag, test)

Introducing the SentenceTransformer Encoding LLM
The all-MiniLM is a very useful model from HuggingFace that maps sentences & paragraphs to 384 dimensional vectors and is ideal for tasks like semantic search.

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

It can run pretty quickly locally.

As an alternative, OpenAI provides a closed-source Embeddings model. Benefits compared to OpenAI embeddings:

It's free and fast!
We can run it locally, so the data never leaves our box - might be useful if you're building a personal RAG

# Pass in a list of texts, get back a numpy array of vectors

vector = encoder.encode(["A proficient AI engineer who has almost reached the finale of AI Engineering Core Track!"])[0]
print(vector.shape)
vector

Let's visualize the vectorized data

In [ ]:
#
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()

In [ ]:
#Calls modal in the cloud and returns a summary of an item from our deployed model on modal
def specialist(item):
    return pricer.price.remote(item.summary)

In [ ]:
#Function to strip out a number when the model returns a text
def get_price(reply):
    reply = reply.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0

Download the Neural Network weights from week 6

In [ ]:
from agents.deep_neural_network import DeepNeuralNetworkInference

runner = DeepNeuralNetworkInference()
runner.setup()
runner.load("deep_neural_network.pth")
#Use the ANN model trained in week 6 to also predict the price of a the given item
def deep_neural_network(item):
    return runner.inference(item.summary)

In [ ]:
#Create an emsemble function that will take the frontier RAG (GPT 5.1 and RAG), the specialist model and the ANN model
#This combines 80% of the output from the frontier RAG, 10% of the specialist model and 10% of the ANN and returns an average price
def ensemble(item):
    price1 = get_price(gpt_5__1_rag(item))
    price2 = specialist(item)
    price3 = deep_neural_network(item)
    return price1 * 0.8 + price2 * 0.1 + price3 * 0.1

In [ ]:
#Apply the above function on the entire test dataset
evaluate(ensemble, test)

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
#Put the notebook into production ready python module
from agents.frontier_agent import FrontierAgent

agent = FrontierAgent(collection)
agent.price("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")

In [ ]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

In [ ]:
from agents.neural_network_agent import NeuralNetworkAgent
agent = NeuralNetworkAgent()

In [ ]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

In [ ]:
from agents.ensemble_agent import EnsembleAgent
agent = EnsembleAgent(collection)

In [ ]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

In [30]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
